# AutoTranscript — Google Colab

Pipeline stable pour transcription française avec **WhisperX 3.8.6 + pyannote community-1**, GPU T4 et **3 locuteurs**.

Cette version utilise **Google Drive** pour conserver l'audio et les résultats même si le runtime Colab est réinitialisé.

Avant de commencer :
- activer un **GPU T4** ;
- accepter l'accès Hugging Face à `pyannote/speaker-diarization-community-1` ;
- créer dans **Secrets** un secret `HF_TOKEN` autorisé pour ce notebook.

Les cellules importantes réimportent elles-mêmes leurs dépendances : il n'est pas nécessaire d'exécuter le test de 3 minutes pour que la transcription complète fonctionne.


In [ ]:
# 1 — Installation des versions figées
import subprocess, sys, os

req = '''whisperx==3.8.6
numpy==2.2.6
pyannote-audio==4.0.7
huggingface-hub==0.36.2
'''
open('/content/requirements-colab.txt', 'w').write(req)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/requirements-colab.txt'])

print('Installation terminée. Redémarrage automatique du noyau...')
os.kill(os.getpid(), 9)


Après le redémarrage automatique, reprendre à la cellule 2. **Ne pas réexécuter la cellule 1** tant que le runtime n'a pas été réinitialisé.


In [ ]:
# 2 — Vérification GPU, versions et token Hugging Face
import os, sys, torch, importlib.metadata as im
from google.colab import userdata

assert torch.cuda.is_available(), "GPU non détecté : active un runtime GPU T4."

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), "HF_TOKEN absent dans Colab > Secrets."
os.environ['HF_TOKEN'] = HF_TOKEN

print('Python', sys.version.split()[0])
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0))
print('whisperx', im.version('whisperx'))
print('pyannote-audio', im.version('pyannote-audio'))
print('numpy', im.version('numpy'))
print('HF_TOKEN : OK')


In [ ]:
# 3 — Monter Google Drive et préparer les dossiers persistants
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/AutoTranscript')
AUDIO_DIR = BASE / 'audio'
RESULTS_DIR = BASE / 'results'

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Dossier audio :', AUDIO_DIR)
print('Dossier résultats :', RESULTS_DIR)
print('
Dépose ton fichier audio dans Mon Drive/AutoTranscript/audio.')


## Choisir le fichier audio

Si `Mon Drive/AutoTranscript/audio` ne contient qu'un seul fichier audio, la cellule suivante le choisit automatiquement.

S'il y en a plusieurs, remplace `AUDIO_NAME = None` par le nom exact, par exemple `AUDIO_NAME = "entretien_01.mp3"`.


In [ ]:
# 4 — Sélection du fichier audio depuis Google Drive
from pathlib import Path

AUDIO_NAME = None  # ex. 'entretien_01.mp3'

extensions = {'.mp3', '.wav', '.m4a', '.mp4', '.aac', '.flac', '.ogg'}
candidates = sorted(
    p for p in AUDIO_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in extensions
)

if AUDIO_NAME:
    AUDIO = AUDIO_DIR / AUDIO_NAME
    assert AUDIO.exists(), f'Fichier introuvable : {AUDIO}'
else:
    assert candidates, f'Aucun fichier audio dans {AUDIO_DIR}'
    assert len(candidates) == 1, (
        'Plusieurs fichiers audio trouvés. Mets leur nom exact dans AUDIO_NAME :
'
        + '
'.join(' - ' + p.name for p in candidates)
    )
    AUDIO = candidates[0]

print('Audio :', AUDIO.name)
print('Taille : %.1f Mo' % (AUDIO.stat().st_size / 1024**2))


In [ ]:
# 5 — Réglages
MODEL = 'medium'
LANGUAGE = 'fr'
NUM_SPEAKERS = 3
BATCH_SIZE = 8

print('Modèle :', MODEL)
print('Langue :', LANGUAGE)
print('Locuteurs :', NUM_SPEAKERS)
print('Batch size :', BATCH_SIZE)


## Test facultatif de 3 minutes

Si la chaîne a déjà été validée, **tu peux sauter cette cellule et passer directement à la transcription complète**.


In [ ]:
# 6 — Test facultatif de 3 minutes
import subprocess, shutil
from pathlib import Path

TEST_START = 0
TEST_DURATION = 180
TEST_AUDIO = Path('/content/test_3min.wav')
TEST_OUT = Path('/content/test_outputs')

shutil.rmtree(TEST_OUT, ignore_errors=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    'ffmpeg', '-y',
    '-ss', str(TEST_START),
    '-i', str(AUDIO),
    '-t', str(TEST_DURATION),
    '-ar', '16000',
    '-ac', '1',
    str(TEST_AUDIO)
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

cmd = [
    'whisperx', str(TEST_AUDIO),
    '--model', MODEL,
    '--language', LANGUAGE,
    '--diarize',
    '--hf_token', HF_TOKEN,
    '--min_speakers', str(NUM_SPEAKERS),
    '--max_speakers', str(NUM_SPEAKERS),
    '--diarize_model', 'pyannote/speaker-diarization-community-1',
    '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--output_dir', str(TEST_OUT),
    '--output_format', 'all'
]

subprocess.check_call(cmd)
print((TEST_OUT / 'test_3min.txt').read_text(encoding='utf-8')[:8000])


## Transcription complète

Les résultats sont écrits directement dans `Mon Drive/AutoTranscript/results/<nom_du_fichier>/`.

Si Colab se déconnecte **après la fin** du traitement, les fichiers sont déjà sauvegardés. Si le runtime est détruit pendant le traitement, l'exécution en cours est interrompue et doit être relancée.


In [ ]:
# 7 — Transcription complète vers Google Drive
import subprocess, time
from pathlib import Path

FULL_OUT = RESULTS_DIR / AUDIO.stem
FULL_OUT.mkdir(parents=True, exist_ok=True)

print('Début de la transcription complète.')
print('Résultats écrits dans :', FULL_OUT)
start_time = time.time()

cmd = [
    'whisperx', str(AUDIO),
    '--model', MODEL,
    '--language', LANGUAGE,
    '--diarize',
    '--hf_token', HF_TOKEN,
    '--min_speakers', str(NUM_SPEAKERS),
    '--max_speakers', str(NUM_SPEAKERS),
    '--diarize_model', 'pyannote/speaker-diarization-community-1',
    '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--output_dir', str(FULL_OUT),
    '--output_format', 'all'
]

subprocess.check_call(cmd)

elapsed = time.time() - start_time
print('
Terminé en %.1f minutes.' % (elapsed / 60))
print('Résultats :')
for p in sorted(FULL_OUT.iterdir()):
    print(' -', p.name)


In [ ]:
# 8 — Produire un TXT horodaté adapté à l'analyse
import json
from pathlib import Path

json_files = sorted(FULL_OUT.glob('*.json'))
assert json_files, 'Aucun JSON WhisperX trouvé.'

json_file = json_files[0]
data = json.loads(json_file.read_text(encoding='utf-8'))

mapping = {}

def hhmmss(x):
    x = int(x)
    h, r = divmod(x, 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

lines = []
for seg in data.get('segments', []):
    raw = seg.get('speaker', 'INCONNU')
    if raw not in mapping:
        mapping[raw] = f'Locuteur {len(mapping)+1}'
    lines.append(
        f"{mapping[raw]} [{hhmmss(seg.get('start', 0))} - {hhmmss(seg.get('end', 0))}] : "
        f"{seg.get('text', '').strip()}"
    )

research_txt = FULL_OUT / (AUDIO.stem + '_recherche.txt')
research_txt.write_text('

'.join(lines), encoding='utf-8')

print('TXT recherche :', research_txt)
print('
Aperçu :
')
print(research_txt.read_text(encoding='utf-8')[:8000])


In [ ]:
# 9 — Optionnel : créer une archive ZIP dans Google Drive
import shutil

archive_base = str(FULL_OUT.parent / (AUDIO.stem + '_transcription'))
archive = shutil.make_archive(archive_base, 'zip', FULL_OUT)

print('Archive créée :', archive)


## En cas de déconnexion Colab

- **Simple reconnexion au même runtime** : ne réinstalle rien ; reprends là où tu en étais.
- **Runtime réinitialisé** : relance la cellule 1, puis après redémarrage les cellules 2 à 5 et enfin la transcription complète.
- L'audio et les résultats déjà écrits restent dans Google Drive.
